In [1]:
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = Path("data")
train_df = pd.read_parquet(DATA / "train_clean.parquet")
test_df  = pd.read_parquet(DATA / "test_clean.parquet")

drop = ["SK_ID_CURR", "TARGET", "CODE_GENDER"]
features = [c for c in train_df.columns if c not in drop]
cat_cols = [c for c in features if not pd.api.types.is_numeric_dtype(train_df[c])]

for c in cat_cols:   # categories learned from TRAIN only
    cats = train_df[c].dropna().unique()
    train_df[c] = pd.Categorical(train_df[c], categories=cats)
    test_df[c]  = pd.Categorical(test_df[c],  categories=cats)

X_tr, X_val, y_tr, y_val = train_test_split(
    train_df[features], train_df["TARGET"],
    test_size=0.1, stratify=train_df["TARGET"], random_state=42)

model = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.03, num_leaves=31,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1)

model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
          callbacks=[lgb.early_stopping(100, verbose=False)])

p_lgb = model.predict_proba(test_df[features])[:, 1]
y_test = test_df["TARGET"]

print("Features:", len(features), "| text features:", len(cat_cols))
print("Trees used:", model.best_iteration_)
print("LightGBM ROC-AUC:", round(roc_auc_score(y_test, p_lgb), 4), "  (baseline 0.7525)")
print("LightGBM PR-AUC: ", round(average_precision_score(y_test, p_lgb), 4), "  (baseline 0.2304)")

C:\Users\RAJA VINAY KUMAR\AppData\Roaming\Python\Python312\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Features: 140 | text features: 14
Trees used: 466
LightGBM ROC-AUC: 0.7708   (baseline 0.7525)
LightGBM PR-AUC:  0.2571   (baseline 0.2304)


In [2]:
import numpy as np

base = pd.read_parquet(DATA / "test_preds_baseline.parquet").set_index("SK_ID_CURR")
p_base = base.loc[test_df["SK_ID_CURR"], "p_baseline"].values
y = y_test.values

rng = np.random.default_rng(42)
diffs = []
for _ in range(200):
    idx = rng.integers(0, len(y), len(y))
    diffs.append(roc_auc_score(y[idx], p_lgb[idx]) - roc_auc_score(y[idx], p_base[idx]))
diffs = np.array(diffs)

print("Average AUC gain:", round(diffs.mean(), 4))
print("95% range:", round(np.percentile(diffs, 2.5), 4), "to", round(np.percentile(diffs, 97.5), 4))
print("LightGBM won in", (diffs > 0).sum(), "of 200 resamples")

Average AUC gain: 0.0181
95% range: 0.0151 to 0.0218
LightGBM won in 200 of 200 resamples


In [3]:
from sklearn.metrics import brier_score_loss

cal = pd.DataFrame({"pred": p_lgb, "actual": y})
cal["group"] = pd.qcut(cal["pred"], 10, labels=range(1, 11))

table = cal.groupby("group", observed=True).agg(
    predicted_pct=("pred", "mean"), actual_pct=("actual", "mean"))
print((table * 100).round(2))

print("\nBrier score LightGBM:", round(brier_score_loss(y, p_lgb), 4))
print("Brier score baseline:", round(brier_score_loss(y, p_base), 4))

       predicted_pct  actual_pct
group                           
1               1.42        1.07
2               2.17        1.74
3               2.85        2.65
4               3.61        3.53
5               4.54        4.13
6               5.72        6.02
7               7.36        7.33
8               9.83       10.26
9              14.21       15.48
10             28.00       28.52

Brier score LightGBM: 0.0671
Brier score baseline: 0.0683


In [4]:
import joblib

Path("models").mkdir(exist_ok=True)
joblib.dump({
    "model": model,
    "features": features,
    "cat_cols": cat_cols,
    "categories": {c: list(train_df[c].cat.categories) for c in cat_cols},
}, "models/credit_model.joblib")

pd.DataFrame({"SK_ID_CURR": test_df["SK_ID_CURR"], "TARGET": y,
              "p_baseline": p_base, "p_lgb": p_lgb}
            ).to_parquet(DATA / "test_preds.parquet", index=False)

print("Model size:", round(Path("models/credit_model.joblib").stat().st_size / 1e6, 2), "MB")

Model size: 1.69 MB


## Notebook 06: LightGBM

LightGBM used 140 features, including 14 text columns. Gender was left out.
It stopped at 466 trees, using 10% of the training data to decide when to stop. The test set stayed untouched.

| | Baseline | LightGBM |
|---|---|---|
| ROC-AUC | 0.7525 | 0.7708 |
| PR-AUC | 0.2304 | 0.2571 |
| Brier score (lower is better) | 0.0683 | 0.0671 |

Is the gain real? I resampled the test set 200 times. LightGBM won all 200, with a 95% range of +0.015 to +0.022 AUC.

Are the probabilities honest? Predicted and actual default rates stayed within 1.3 points in every risk group. The riskiest 10% defaulted at 28.5%, the safest 10% at 1.1%.